# {{NN}} — Pull {{SOURCE_NAME}}

**Source:** {{SOURCE_NAME}}  
**Provider:** {{PROVIDER}}  
**URL:** {{SOURCE_URL}}  
**Cadence:** {{CADENCE}}  

## What this notebook does
{{SOURCE_DESCRIPTION}}

## Required environment variables
```
ADLS_ACCOUNT_NAME — Azure storage account name
ADLS_CONTAINER    — Container name (default: 'data')
```


In [ ]:
import io
import os
import re
import requests
import pandas as pd
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

SOURCE_KEY        = "{{SOURCE_KEY}}"
SOURCE_URL        = "{{SOURCE_URL}}"

print(f"ADLS target : abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/raw/{SOURCE_KEY}/")
print(f"Run date    : {RUN_DATE}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## ISO3 mapping helper

If `{{SOURCE_NAME}}` already publishes 3-letter ISO codes, you can delete this cell and the ISO3-mapping step in `cell-clean`. Keep it whenever the source uses country names — the BTI integration broke downstream because we forgot this step.

In [ ]:
# Inline override map for known edge cases. Extend as needed.
_NAME_OVERRIDES = {
    "cote d'ivoire": "CIV", "ivory coast": "CIV", "côte d'ivoire": "CIV",
    "eswatini": "SWZ", "swaziland": "SWZ",
    "czechia": "CZE", "czech republic": "CZE",
    "cabo verde": "CPV", "cape verde": "CPV",
    "congo, dem. rep.": "COD", "democratic republic of the congo": "COD", "dr congo": "COD",
    "congo, rep.": "COG", "republic of the congo": "COG",
    "egypt, arab rep.": "EGY",
    "gambia, the": "GMB", "the gambia": "GMB",
    "yemen, rep.": "YEM",
    "iran, islamic rep.": "IRN",
    "korea, rep.": "KOR", "korea, dem. people's rep.": "PRK",
    "russian federation": "RUS",
    "syrian arab republic": "SYR",
    "tanzania": "TZA", "united republic of tanzania": "TZA",
    "viet nam": "VNM", "vietnam": "VNM",
    "venezuela, rb": "VEN",
    "turkiye": "TUR", "türkiye": "TUR", "turkey": "TUR",
    "micronesia, fed. sts.": "FSM",
    "st. lucia": "LCA", "st. vincent and the grenadines": "VCT", "st. kitts and nevis": "KNA",
}

try:
    import pycountry
    _HAS_PYCOUNTRY = True
except ImportError:
    _HAS_PYCOUNTRY = False
    print("WARNING: pycountry not installed — relying on _NAME_OVERRIDES alone.")

def name_to_iso3(name) -> str | None:
    if not isinstance(name, str) or not name.strip():
        return None
    raw_key  = name.lower().strip()
    norm_key = re.sub(r"[^a-z0-9 ]", "", raw_key)
    if raw_key in _NAME_OVERRIDES:
        return _NAME_OVERRIDES[raw_key]
    if norm_key in _NAME_OVERRIDES:
        return _NAME_OVERRIDES[norm_key]
    if _HAS_PYCOUNTRY:
        try:
            return pycountry.countries.lookup(name).alpha_3
        except LookupError:
            return None
    return None

## Fetch raw data

**TODO** — replace the placeholder body with the real download / API call. The placeholder raises `NotImplementedError` on purpose so the notebook fails loudly until customised.

In [ ]:
def fetch_{{SOURCE_KEY}}() -> pd.DataFrame:
    """Fetch raw data from {{SOURCE_NAME}}. Replace this body.

    Should return a DataFrame with, at minimum, a country identifier
    (`iso3` or `country_name`) and `year`. Sub-sector / variable columns
    that need transforms downstream should already be in their final
    snake_case names here — `02/03` ENG_CFG references these names
    verbatim, so name drift is silent and bites later.
    """
    raise NotImplementedError(
        "fetch_{{SOURCE_KEY}}() not implemented — replace with real download. "
        f"See {SOURCE_URL} for the source."
    )

raw_df = fetch_{{SOURCE_KEY}}()
print(f"Fetched: shape={raw_df.shape}, columns={list(raw_df.columns)[:10]}")

## Clean and reshape

Keep this cell focused on: column lower-casing, type coercion, `year` extraction, and ISO3 mapping. Sub-sector aggregations belong in a separate cell if they get long.

In [ ]:
df = raw_df.copy()
df.columns = [c.lower().strip() for c in df.columns]

# Year coercion — `02/02` requires `year` to be Int64.
if "year" not in df.columns:
    raise ValueError("raw data missing `year` column — extract or rename before this point")
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

# ISO3 mapping (skip if source already provides iso3).
if "iso3" not in df.columns:
    name_col = next((c for c in df.columns if c in ("country_name", "country")), None)
    if name_col is None:
        raise ValueError(
            "raw data missing both `iso3` and a `country_name` / `country` column — "
            "cannot key the panel"
        )
    df["iso3"] = df[name_col].apply(name_to_iso3)
    n_unmatched = df["iso3"].isna().sum()
    if n_unmatched:
        unmatched_names = sorted(df.loc[df["iso3"].isna(), name_col].dropna().unique())[:10]
        print(f"WARNING: {n_unmatched} rows lacked an ISO3 match. Examples: {unmatched_names}")
        print("  → either extend _NAME_OVERRIDES in cell-iso3 or accept the drop in the validate step.")

# Move iso3 + year to the front for readability.
front = ["iso3", "year"]
df = df[front + [c for c in df.columns if c not in front]]
print(f"After clean: shape={df.shape}")
df.head(3)

## Validate

In [ ]:
before = len(df)
df = df.dropna(subset=["iso3", "year"]).drop_duplicates(["iso3", "year"]).reset_index(drop=True)
after = len(df)
if before != after:
    print(f"Dropped {before - after:,} rows (missing iso3/year or duplicate keys)")

assert df["iso3"].notna().all(),                 "iso3 has NaNs after dropna — check name mapping"
assert df["year"].notna().all(),                 "year has NaNs after dropna"
assert df.duplicated(["iso3", "year"]).sum() == 0, "duplicate (iso3, year) rows survived"

print(f"Validated panel: {len(df):,} rows × {df.shape[1]} cols")
print(f"Countries  : {df['iso3'].nunique()}")
print(f"Year range : {df['year'].min()}–{df['year'].max()}")

## Write to ADLS

In [ ]:
write_parquet(df, f"raw/{SOURCE_KEY}/{RUN_DATE}/{SOURCE_KEY}_panel.parquet")

## Summary

In [ ]:
print("=" * 55)
print("{{SOURCE_NAME}} pull complete")
print("=" * 55)
print(f"  Rows (country-years) : {len(df):,}")
print(f"  Countries            : {df['iso3'].nunique()}")
print(f"  Year range           : {df['year'].min()}–{df['year'].max()}")
feat_cols = [c for c in df.columns if c not in ('iso3', 'year')]
print(f"  Features             : {feat_cols}")
print()
print("ADLS path written:")
print(f"  raw/{SOURCE_KEY}/{RUN_DATE}/{SOURCE_KEY}_panel.parquet")